In [1]:

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import mlflow
import optuna
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path

/home/bashar/anaconda3/envs/mlops/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df_path = Path('data/df_preprocessed.csv')
if df_path.exists():
    df = pd.read_csv(df_path)
else:
    print(f"No such file : {df_path}")

xgboost_cat = {-1:2, 0:0, 1:1}
df['Sentiment'] = df['Sentiment'].apply(lambda x:xgboost_cat[x])


In [3]:
def objective(trial:optuna.Trial):
    vectorizer = TfidfVectorizer(ngram_range=(1,3), max_features=100)
    X = vectorizer.fit_transform(df['CommentText']).toarray()[:10000]
    y = df['Sentiment'][:10000]

    X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
    n_estimators = trial.suggest_int('n_estimator',2, 5)
    max_depth = trial.suggest_int('max_depth', 2, 5)
    learning_rate = trial.suggest_loguniform('lr', 1e-3, 1)


    model = XGBClassifier(n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate, objective='binary:logistic')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc



In [4]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)


[I 2026-09-23 16:35:59,311] A new study created in memory with name: no-name-12a900b0-c9fa-4bd4-8e77-4f3252cf874c
/tmp/ipykernel_929941/1849139798.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-3, 1)
[I 2026-09-23 16:36:46,448] Trial 0 finished with value: 0.333 and parameters: {'n_estimator': 3, 'max_depth': 5, 'lr': 0.005032263588696955}. Best is trial 0 with value: 0.333.
/tmp/ipykernel_929941/1849139798.py:9: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  learning_rate = trial.suggest_loguniform('lr', 1e-3, 1)
[I 2026-09-23 16:37:33,393] Trial 1 finished with value: 0.332 and parameters: {'n_estimator': 5, 'max_d

FrozenTrial(number=6, state=<TrialState.COMPLETE: 1>, values=[0.5025], datetime_start=datetime.datetime(2026, 9, 23, 16, 40, 39, 624868), datetime_complete=datetime.datetime(2026, 9, 23, 16, 41, 24, 199949), params={'n_estimator': 4, 'max_depth': 4, 'lr': 0.9179341326470604}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimator': IntDistribution(high=5, log=False, low=2, step=1), 'max_depth': IntDistribution(high=5, log=False, low=2, step=1), 'lr': FloatDistribution(high=1.0, log=True, low=0.001, step=None)}, trial_id=6, value=None)